# Practice 04 — Deep Learning from Scratch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/04-deep-learning.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/04-deep-learning.ipynb)

Companion drills for **Phase 4** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with the [neural-net playground](https://learn-by-visuallization.org/illustrated/4-deep-learning/nn-playground.html),
[activations](https://learn-by-visuallization.org/illustrated/4-deep-learning/activations.html) and the scrub-through
[backpropagation explainer](https://learn-by-visuallization.org/illustrated/4-deep-learning/backprop.html).
Pure NumPy — you'll build the pieces PyTorch automates.

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup, no GPU needed).

In [ ]:
NEEDED = [("numpy", "numpy")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — activations & their slopes
Implement `sigmoid`, `relu`, and their derivatives. The derivatives are what backprop multiplies —
get them wrong and nothing downstream can learn.

*Hint: σ′(z) = σ(z)·(1−σ(z)); relu′(z) = 1 where z>0 else 0.*

In [ ]:
def sigmoid(z):
    return None  # TODO

def d_sigmoid(z):
    return None  # TODO

def relu(z):
    return None  # TODO

def d_relu(z):
    return None  # TODO


In [ ]:
z = np.array([-2.0, 0.0, 3.0])
check("sigmoid values", lambda: np.allclose(sigmoid(z), 1 / (1 + np.exp(-z))), "1/(1+e^-z)")
check("sigmoid slope peaks at 0.25", lambda: abs(d_sigmoid(np.array([0.0]))[0] - 0.25) < 1e-9,
      "σ'(0) = 0.5·0.5 — the flat shoulders elsewhere cause vanishing gradients")
check("relu kills negatives only", lambda: np.allclose(relu(z), [0, 0, 3]), "max(0, z)")
check("relu slope is a step", lambda: np.allclose(d_relu(z), [0, 0, 1]), "(z > 0) as floats")

### Exercise 2 — one neuron's backward pass, by hand
The backprop explainer's exact neuron: `z = w·x + b`, `a = σ(z)`, `L = (a − y)²`.
Compute the three gradients with the chain rule (no autograd!).

*Hint: ∂L/∂a = 2(a−y); ∂L/∂z = ∂L/∂a · σ′(z); ∂L/∂w = ∂L/∂z · x; ∂L/∂b = ∂L/∂z.*

In [ ]:
w, x, b, y = 0.8, 1.5, -0.5, 1.0
z = w * x + b
a = 1 / (1 + np.exp(-z))
L = (a - y) ** 2

dL_dw = None  # TODO
dL_db = None  # TODO


In [ ]:
_ga = 2 * (a - y); _gz = _ga * a * (1 - a)
check("dL/dw via the chain rule", lambda: abs(dL_dw - _gz * x) < 1e-9,
      "multiply local slopes along the path: 2(a-y) · σ'(z) · x")
check("dL/db is the upstream signal", lambda: abs(dL_db - _gz) < 1e-9,
      "the +b edge has local slope 1")

### Exercise 3 — solve XOR with a hidden layer
`Xor` is the classic 4-point dataset no single line can separate. Finish `train_xor`: a 2-4-1 MLP
trained with full-batch gradient descent. The forward/backward math is written — you supply the
**update step** (the one line gradient descent actually is).

*Hint: every parameter P gets `P -= lr * gP`.*

In [ ]:
Xor = np.array([[0,0],[0,1],[1,0],[1,1]], float)
Yor = np.array([[0],[1],[1],[0]], float)

def train_xor(lr=1.0, epochs=3000, seed=1):
    r = np.random.default_rng(seed)
    W1, b1 = r.normal(0, 1, (2, 4)), np.zeros(4)
    W2, b2 = r.normal(0, 1, (4, 1)), np.zeros(1)
    sig = lambda t: 1 / (1 + np.exp(-t))
    for _ in range(epochs):
        # forward
        h = sig(Xor @ W1 + b1)
        out = sig(h @ W2 + b2)
        # backward (already derived for you)
        d_out = (out - Yor) * out * (1 - out)
        gW2, gb2 = h.T @ d_out, d_out.sum(0)
        d_h = (d_out @ W2.T) * h * (1 - h)
        gW1, gb1 = Xor.T @ d_h, d_h.sum(0)
        # TODO: the gradient descent update for W1, b1, W2, b2
        ...
    return sig(sig(Xor @ W1 + b1) @ W2 + b2)


In [ ]:
check("the MLP solves XOR (all 4 points within 0.2 of target)",
      lambda: np.all(np.abs(train_xor() - Yor) < 0.2),
      "P -= lr * gP for each of W1, b1, W2, b2 — without the update, nothing learns")

### Exercise 4 — epochs, batches, iterations
Fill in the arithmetic every training log makes you do: `n=50,000` samples, `batch=128`.
How many **iterations per epoch** (round up), and how many **total iterations** for 10 epochs?

*Hint: ceil(n / batch) — the last, smaller batch still counts.*

In [ ]:
n, batch, epochs = 50_000, 128, 10

iters_per_epoch = None  # TODO
total_iters     = None  # TODO


In [ ]:
import math
check("iterations per epoch (round UP)", lambda: iters_per_epoch == math.ceil(50_000 / 128),
      "391 — the 79-sample leftover batch is still an iteration")
check("total iterations", lambda: total_iters == iters_per_epoch * 10, "per-epoch × epochs")

### Stretch goal — watch it overfit
Shrink XOR training to 200 epochs vs 3000 and compare outputs; then add a 5th noisy point and see
what the tiny network does. Relate to the
[bias–variance explainer](https://learn-by-visuallization.org/illustrated/3-classic-ml/bias-variance.html).

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")


<!--APPLY_SECTION-->
---
## 🧩 Apply — practice problems  🔒
You've learned the ideas above. Now prove it. Implement each function and run its
`grade(...)` — the tests are **hidden**, so a ✅ means it's genuinely correct. Stuck? call
`hint('pN')`. Full solutions are **locked in the Appendix** at the very bottom.

In [ ]:
# --- Apply autograder (hidden tests) ---
import base64 as _b64, marshal as _msh
def _eq(a,b,tol=1e-6):
    if isinstance(b,(list,tuple)): return isinstance(a,(list,tuple)) and len(a)==len(b) and all(_eq(x,y,tol) for x,y in zip(a,b))
    if isinstance(b,dict): return isinstance(a,dict) and a.keys()==b.keys() and all(_eq(a[k],b[k],tol) for k in b)
    if isinstance(b,bool): return a is b or a==b
    if isinstance(b,float):
        try: return abs(a-b)<=tol
        except Exception: return False
    return a==b
_TESTS={"p1": "WwMAAAApAqkBWwMAAADp/v///+kAAAAA6QMAAABbAwAAAHICAAAAcgIAAAByAwAAACkCqQFbAgAAAOkBAAAA6f////9bAgAAAHIFAAAAcgIAAAApAqkBWwIAAAByAgAAAHICAAAAWwIAAAByAgAAAHICAAAA", "p2": "WwMAAAApAqkB6QAAAABnAAAAAAAA4D8pAqkB6QIAAABnyXmKWn0v7D8pAqkB6f7///9nrjGsKxWEvj8="}
_HINTS={"p1": ["For each x return x if x>0 else 0."], "p2": ["Use math.exp(-x) in the denominator.", "sigmoid(0) is exactly 0.5."]}
_SOLUTIONS={}
def grade(pid, fn):
    cs=_msh.loads(_b64.b64decode(_TESTS[pid]))
    for k,(args,exp) in enumerate(cs,1):
        try: got=fn(*args)
        except Exception as e:
            print(f"❌ {pid}: hidden case {k}/{len(cs)} raised {type(e).__name__}: {e}. Try hint('{pid}')."); return False
        if not _eq(got,exp):
            print(f"❌ {pid}: hidden case {k}/{len(cs)} failed — not right yet. Try hint('{pid}')."); return False
    print(f'✅ {pid}: all {len(cs)} hidden tests passed 🎉'); return True
def hint(pid,n=None):
    hs=_HINTS.get(pid,[]); rng=range(len(hs)) if n is None else [n-1]
    [print(f'Hint {i+1}: {hs[i]}') for i in rng]
def reveal(pid):
    if pid not in _SOLUTIONS: print('🔒 Locked — run the Appendix cell at the bottom first.'); return
    print(_b64.b64decode(_SOLUTIONS[pid]).decode())
print('Apply autograder ready ✓')


### p1 · `relu(xs)`
Apply ReLU elementwise to a list: negatives -> 0.

In [ ]:
def relu(xs):
    # TODO: max(0, x) for each
    pass

grade('p1', relu)
# hint('p1')


### p2 · `sigmoid(x)`
Logistic sigmoid 1/(1+e^-x) of a single float.

In [ ]:
import math
def sigmoid(x):
    # TODO
    pass

grade('p2', sigmoid)
# hint('p2')


### ✅ Score

In [ ]:
import io, contextlib
_pairs=[('p1', relu), ('p2', sigmoid)]
done=0
for pid, fn in _pairs:
    buf=io.StringIO()
    with contextlib.redirect_stdout(buf): done+=grade(pid, fn)
print(f'\nApply score: {done} / 2' + ('  — nailed it 🏆' if done==2 else ''))


<!--APPLY_SECTION-->
### Appendix — Solutions (locked 🔒)
Try first. Run this to unlock, then `reveal('pN')`.

In [ ]:
_SOLUTIONS={"p1": "ZGVmIHJlbHUoeHMpOgogICAgcmV0dXJuIFt4IGlmIHg+MCBlbHNlIDAgZm9yIHggaW4geHNd", "p2": "aW1wb3J0IG1hdGgKZGVmIHNpZ21vaWQoeCk6CiAgICByZXR1cm4gMS8oMSttYXRoLmV4cCgteCkp"}
print('🔓 Unlocked. reveal("p1") / reveal("p2").')
